# Demo of the nanodent package

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import numpy as np

import nanodent

# Load a study from a folder

In [ ]:
# Load study by passing foldet with experimental data
study = nanodent.load_folder("../tests/data/")

# Filter out experiments with low quality
study = study.classify_quality()

In [ ]:
# Manually disable experiments
study = study.disable_experiments(
  [
      "experiment_c",
  ],
  reason="manual",
)

# Group experiments

In [ ]:
# Show timeline: grouped by maximum time gap between the experiments
timeline_fig, timeline_ax = nanodent.plot_group_timeline(study, max_gap=timedelta(minutes=200))

In [ ]:
# Or group manually by datetime ranges
datetime_ranges = [
        (
            datetime(2026, 3, 3, 0, 0, 0),
            datetime(2026, 3, 5, 0, 0, 0),
        ),
        (
            datetime(2026, 3, 5, 0, 0, 1),
            datetime(2026, 3, 12, 0, 0, 0),
        ),
]
groups = study.group_by_datetime_ranges(datetime_ranges)
timeline_fig, timeline_ax = nanodent.plot_group_timeline(groups, study=study)

# Do analyses on study's enabled experiments

## Analyses on all (enabled) experiments

In [ ]:
# Define Savitzky-Golay filter params: window length and polynomial order
smoothing = {"window_length": 11, "polyorder": 1}

# Detect unloading curve
study = study.detect_unloading()

# Detect onset
study = study.detect_onset(
    baseline_start_index=75,
    baseline_end_index=300,
    consecutive=4,
    k=5,
    overwrite=True,
)

# Detect force peaks
study = study.detect_force_peaks(overwrite=True)

# Run Oliver-Pharr analysis on all enabled experiment
study = study.analyze_oliver_pharr(
    fit_model="power_law_full",
    epsilon=0.75,
    include_disabled=False,
    overwrite=True,
    #smoothing=smoothing,
)

_ = nanodent.save_experiment_plots(
    study,
    output_dir = "./output",
    fit_kwargs={"color": "lime", "alpha": .9},
    unloading_kwargs={"color": "lightgray"},
    color='k',
)

## Analyses on selected experiments

In [ ]:
overwrite = True

# Manually recompute for selected experiments
manual = [
    "experiment_c",
]

# Define Savitzky-Golay filter params: window length and polynomial order
smoothing = {"window_length": 51, "polyorder": 1}
#smoothing = None


# Detect onset of all (enabled) experiments of the study
study = study.detect_onset(
    stems=manual,
    baseline_start_index=75,
    baseline_end_index=300,
    consecutive=5,
    k=20,
    overwrite=True,
    smoothing=smoothing
)

# Detect force peaks of all (enabled) experiments of the study
study = study.detect_force_peaks(
    prominence=50,
    stems=manual,
    overwrite=overwrite,
    threshold=.01,
)

# Run Oliver-Pharr analysis on all (enabled) experiment of the study
study = study.analyze_oliver_pharr(
    stems=manual,
    epsilon=0.75,
    include_disabled=False,
    overwrite=overwrite,
    smoothing=smoothing,
    unloading_fraction=0.25,
)

_ = nanodent.save_experiment_plots(
    study.get_experiments(stems=manual),
    output_dir = "./output",
    fit_kwargs={"color": "lime", "alpha": .9},
    color='darkgray',
)

# Visualization

In [ ]:
# Illustrate experiments
fig, ax = plt.subplots()
nanodent.plot_experiments(
    ax,
    study.experiments,
    fit_kwargs={"color": "gray", "linestyle": "solid", "linewidth": "2"},
    smoothing=smoothing,
    zero_onset=True,
    x="disp_nm",
    y="force_uN",
    cmap="rainbow",
    marker=".",
    linestyle=None
)

ax.set_xbound(lower=0., upper=None)
ax.set_ybound(lower=0., upper=None)
ax.set_xlabel("Displacement h / nm")
ax.set_ylabel("Force P / μN")

In [ ]:
# Illustrate some experiments
fig = plt.figure(figsize=(7.22433,4))
gs = gridspec.GridSpec(nrows=1, ncols=2, hspace=.5, wspace=.5)

for g,G in enumerate(groups):
    ax = fig.add_subplot(gs[g])
    ax = nanodent.plot_experiments(
        ax,
        G,
        study=study,
        fit_kwargs={"color": "gold", "linestyle": "solid", "linewidth": "2"},
        zero_onset=True,
        x="disp_nm",
        y="force_uN",
        cmap="rainbow",
    )
    ax.set_xbound(lower=0., upper=None)
    ax.set_ybound(lower=0., upper=None)
    ax.set_xlabel("Displacement h / nm")
    ax.set_ylabel("Force P / μN")
fig.savefig("test.png")

# Illustrate per-time scalar data

In [ ]:
rows_hardness = study.scalar_series("hardness")
x_hardness = [row["timestamp"] for row in rows_hardness]
y_hardness = [row["value"] for row in rows_hardness]

rows_modulus = study.scalar_series("reduced_modulus")
x_modulus = [row["timestamp"] for row in rows_modulus]
y_modulus = [row["value"] for row in rows_modulus]

In [ ]:
grouped_hardness = study.group_scalar_series(
    groups=groups,
    metric="hardness",
)
grouped_modulus = study.group_scalar_series(
    groups=groups,
    metric="reduced_modulus",
)

In [ ]:
# Illustrate some experiments
fig = plt.figure(figsize=(5,5))
gs = gridspec.GridSpec(nrows=2, ncols=1, hspace=.1)

ax0 = fig.add_subplot(gs[0])
ax0.grid(axis='x')
ax0.plot(x_hardness, y_hardness, marker="o", linestyle="-", color='k')
for g, G in enumerate(grouped_hardness):
    print(G["value"])
    ax0.errorbar([G["timestamp"]], [G["value"]], yerr=[G["std"]/np.sqrt(G["count"])], marker='.', color='gold', markersize=10)
ax0.set_xlabel("Acquisition Time")
ax0.set_ylabel("Hardness")

ax1 = fig.add_subplot(gs[1])
ax1.grid(axis='x')
ax1.plot(x_modulus, y_modulus, marker="o", linestyle="-", color='k')
for g, G in enumerate(grouped_modulus):
    print(G["value"])
    ax1.errorbar([G["timestamp"]], [G["value"]], yerr=[G["std"]/np.sqrt(G["count"])], marker='.', color='gold', markersize=10)
ax1.set_xlabel("Acquisition Time")
ax1.set_ylabel("Reduced Modulus")

fig.autofmt_xdate()

# Save study state for later reuse

In [ ]:
study.save_session("study-analysis.pkl")